# 🧠 UMBRA — Google Colab Training Notebook

**Model:** `HuggingFaceTB/SmolLM-135M` (4-bit QLoRA)  
**Trainer:** GRPO via TRL  
**Datasets:** hh-rlhf · pii-masking-300k · truthful_qa · toxic-chat · daily_dialog  

**Runtime:** T4 GPU (16 GB) — set via *Runtime → Change runtime type → T4*

---
### Steps
1. Run **Cell 1** to install dependencies  
2. Run **Cell 2** to mount Google Drive and upload your project  
3. Run **Cell 3** to verify the GPU  
4. Run **Cell 4** to start training  
5. Run **Cell 5** to evaluate after training  

In [ ]:
# ── Cell 1: Install all dependencies ──────────────────────────────────────────
!pip install -q \
    "trl>=0.9.0" \
    "datasets>=2.18.0" \
    "transformers>=4.40.0" \
    "bitsandbytes>=0.43.0" \
    "peft>=0.10.0" \
    "accelerate>=0.29.0" \
    "gymnasium>=0.29.0" \
    "matplotlib>=3.8" \
    "numpy>=1.26" \
    "fastapi" \
    "uvicorn"

print('✅ All dependencies installed.')

In [ ]:
# ── Cell 2: Mount Google Drive and set project path ───────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os

# ⚠️  Change this path to wherever you uploaded the Umbra folder in your Drive
PROJECT_PATH = '/content/drive/MyDrive/Umbra'

assert os.path.isdir(PROJECT_PATH), (
    f'❌ Project folder not found at {PROJECT_PATH}. '
    'Upload your Umbra folder to Google Drive and update PROJECT_PATH above.'
)

os.chdir(PROJECT_PATH)
print(f'✅ Working directory: {os.getcwd()}')
print('Files:', os.listdir('.'))

In [ ]:
# ── Cell 3: Verify GPU + fix Python path + smoke test ─────────────────────────
import sys, os, torch

# ── CRITICAL: add project root to sys.path so all UMBRA modules resolve ───────
# This is the fix for: ModuleNotFoundError: No module named 'reward'
PROJECT_PATH = os.getcwd()   # already set to /content/drive/MyDrive/Umbra by Cell 2
if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)
print(f'✅ sys.path[0] = {sys.path[0]}')

# ── GPU check ─────────────────────────────────────────────────────────────────
if not torch.cuda.is_available():
    raise RuntimeError(
        '❌ No GPU detected! Go to Runtime → Change runtime type → T4 GPU and restart.'
    )
gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'✅ GPU: {gpu_name}  |  VRAM: {vram_gb:.1f} GB')

# ── Full import sanity check ───────────────────────────────────────────────────
from reward.reward_model      import ShapedRewardModel
from sentrix.cialdini_stress  import run_cialdini_stress
from shadow.shadow_agent      import ShadowAgent
from shadow.arms_race_trainer import ArmsRaceTrainer
from demo.graph_generator     import generate_all_graphs
from data.dataset_loader      import load_hh_rlhf, build_grpo_prompts
print('✅ All UMBRA modules imported successfully.')

# ── Dataset loader quick smoke test ───────────────────────────────────────────
sample_ds      = load_hh_rlhf(split='train', max_samples=5)
sample_prompts = build_grpo_prompts(sample_ds, max_prompts=3)
print(f'✅ Dataset loader OK — sample prompt: {sample_prompts[0][:80]}…')
print('\n🚀 Ready to train! Run Cell 4.')

In [ ]:
# ── Cell 4: Run full UMBRA training ───────────────────────────────────────────
# Full pipeline (~2h 30min on T4 GPU):
#   1. Load 5 HuggingFace datasets            (cached after first run)
#   2. Benchmark Sentrix PII guard             → logs/sentrix_benchmark.json
#   3. Capture BEFORE metrics (baseline)       → logs/before_metrics.json
#   4. GRPO fine-tune SmolLM-135M             (~45 min) → umbra_grpo_ckpt/
#   5. Shadow Arms Race (2 rounds × 50 eps)   (~15 min) → logs/arms_race_log.jsonl
#   6. RL Loop 500 eps ShapedRewardModel      (~30 min) → logs/rollout_samples.jsonl
#   7. Cialdini Stress Test (6 × 10 eps)      (~10 min) → logs/cialdini_results.json
#   8. AFTER metrics + 4 graphs generated              → logs/reward_graphs/
#   9. Save final model                                → umbra_final/
#
# ── To run in FAST MODE (~5 min), uncomment these lines BEFORE running: ───────
# import re, pathlib
# src = pathlib.Path('train.py').read_text()
# src = re.sub(r'USE_GRPO\s*=\s*True',  'USE_GRPO  = False', src)
# src = re.sub(r'USE_SHADOW\s*=\s*True','USE_SHADOW = False', src)
# src = re.sub(r'EPISODES\s*=\s*500',   'EPISODES   = 50',   src)
# src = re.sub(r'CIALDINI_EPS\s*=\s*10','CIALDINI_EPS = 3',  src)
# pathlib.Path('train_fast.py').write_text(src)
# %run train_fast.py
# ─────────────────────────────────────────────────────────────────────────────

%run train.py

In [ ]:
# ── Cell 5: Evaluate trained model ────────────────────────────────────────────
%run evaluate.py

In [ ]:
# ── Cell 6: View generated training graphs ────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

graph_dir = Path('logs/reward_graphs')
graphs    = sorted(graph_dir.glob('*.png'))

if not graphs:
    print('No graphs found yet — run Cell 4 (train.py) first.')
else:
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    fig.patch.set_facecolor('#0d1117')
    for ax, path in zip(axes.flat, graphs):
        img = mpimg.imread(str(path))
        ax.imshow(img)
        ax.set_title(path.stem.replace('_', ' ').title(),
                     color='white', fontsize=13, pad=8)
        ax.axis('off')
    # Hide unused subplot if fewer than 4 graphs
    for ax in axes.flat[len(graphs):]:
        ax.set_visible(False)
    plt.tight_layout()
    plt.show()
    print(f'✅ Displayed {len(graphs)} graphs from {graph_dir}/')

In [ ]:
# ── Cell 7 (optional): Download model + logs to local machine ─────────────────
import shutil, os
from google.colab import files

# Bundle the trained model
if os.path.isdir('umbra_final'):
    shutil.make_archive('umbra_final', 'zip', 'umbra_final')
    files.download('umbra_final.zip')
    print('✅ Model download started.')
else:
    print('⚠ umbra_final/ not found — run Cell 4 first.')

# Bundle logs (graphs + metrics + Cialdini results)
if os.path.isdir('logs'):
    shutil.make_archive('umbra_logs', 'zip', 'logs')
    files.download('umbra_logs.zip')
    print('✅ Logs download started.')
else:
    print('⚠ logs/ not found.')

In [ ]:
# ── Cell 7 (optional): View Sentrix benchmark report ─────────────────────────
import json

with open('logs/sentrix_benchmark.json') as f:
    report = json.load(f)

print('=== Sentrix PII Guard Benchmark (vs ai4privacy/pii-masking-300k) ===')
print(f"  True Positives  : {report['true_positives']}")
print(f"  False Positives : {report['false_positives']}")
print(f"  False Negatives : {report['false_negatives']}")
print(f"  Precision       : {report['precision']}")
print(f"  Recall          : {report['recall']}")
print(f"  F1 Score        : {report['f1']}")